# Appendix C2. Disaggregated Sub-Indices: Sensitivity of LSTM Results to Random Initialization

In [ ]:
# --- ENV FIRST ---
import os
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"
# --- IMPORTS ---
import random
import numpy as np
import tensorflow as tf
# --- TF CONFIG ---
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.set_visible_devices([], 'GPU')
# --- RESET ---
tf.keras.backend.clear_session()

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import recall_score, f1_score, roc_auc_score, confusion_matrix, accuracy_score, roc_curve
import itertools
from sklearn.utils.class_weight import compute_class_weight
from tensorflow import keras
from tensorflow.keras import layers
try:
  import keras_tuner as kt
except:
  !pip install keras-tuner
  import keras_tuner as kt
from google.colab import files
from google.colab import drive
import pandas as pd
import io
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 1.1 MB/s eta 0:00:00


Build a function to import data to the notebook.

In [ ]:
def import_data(file_path):
  try:
    drive.mount('/content/drive', force_remount=True)
    # Check if file exists
    if os.path.exists(file_path):
      df = pd.read_parquet(file_path)
      print(f"Loaded dataframe from Drive ({file_path})")
    else:
      raise FileNotFoundError(f"File not found at {file_path}")

  except Exception as e:
    print(f"Drive not available or file missing: {e}")
    print("Please upload dataframe manually.")
    uploaded = files.upload()

    # Automatically read the uploaded file
    file_name = list(uploaded.keys())[0]  # pick the first uploaded file
    try:
      df = pd.read_parquet(io.BytesIO(uploaded[file_name]))
    except:
      print("Wrong file extension. Parquet file required.")
    print(f"Loaded {file_name} from manual upload.")
    return df

Build a function to fit a given model on training data and output out-of-sample predictions, class probabilities, as well as the following metrics estimated on test data:
- Recall
- F1 score
- ROC AUC
- Confusion Matrix.

In [ ]:
def train_classifier(model, X_train, y_train, X_test, y_test, X_val=False, y_val=False, threshold=0.5, early_stopping=False):
  is_keras = hasattr(model, "fit") and hasattr(model, "predict") and not hasattr(model, "predict_proba")
  if is_keras:
    if early_stopping:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False, callbacks=[combined_metric, es])
    else:
      model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size, class_weight=class_weight, validation_data=(X_val, y_val), shuffle=False)
    y_score = model.predict(X_test).ravel()
    y_pred = (y_score >= threshold).astype(int)
  else:
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]

  # Output Following Metrics:
  recall = recall_score(y_test, y_pred)
  f1 = f1_score(y_test, y_pred)
  roc_auc = roc_auc_score(y_test, y_score)
  cm = confusion_matrix(y_test, y_pred)

  return recall, f1, roc_auc, cm, y_score, y_pred

Import the df_sse.parquet file, containing the Systemic Stress Events (the target variable) data, available in the <a href='https://github.com/SebastianoDenegri/vietnam-systemic-risk-index/tree/main/data/machine_learning'>data/machine_learning</a> subdirectory.

In [ ]:
file_path = "..."
df_sse = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving df_sse.parquet to df_sse.parquet
Loaded df_sse.parquet from manual upload.


Import the parquet file with the 5 disaggregated sub-indices already normalized via ECDF, risk_ecdf.parquet, available in the <a href='https://github.com/SebastianoDenegri/vietnam-systemic-risk-index/tree/main/data/machine_learning'>data/machine_learning</a> subdirectory.

In [ ]:
risk_ecdf = import_data(file_path)

Drive not available or file missing: Error: credential propagation was unsuccessful
Please upload dataframe manually.


Saving risk_ecdf.parquet to risk_ecdf.parquet
Loaded risk_ecdf.parquet from manual upload.


Build a dataframe with the 5 disaggregate indicators (predictors) and the Systemic Stress Events variable (target).  
The data sets are defined as follows:
- Training data: until 1st March 2023 (excluded)
- Validation data: from 1st March 2023 until 31st March 2024
- Test data: from 1st April 2024  

We apply a 20-day purge period between training and validation as well as validation and test sets.  
Sequence length: 60 days (a quarter of trading activities).

In [ ]:
# build a dataframe with the 5 disaggregate indicators (predictors) and the Systemic Stress Events variable (target)
df_bench = pd.concat([risk_ecdf, df_sse['systemic_stress_event']], axis=1, join='inner').dropna()
# align the dataframe at the same starting date of the VSRI
df_bench = df_bench.loc['2017-01-03':].dropna()

In [ ]:
df_bench.shape

(2285, 6)

In [ ]:
# split data in training and test set and apply purging
split_date = "2024-03-31"
purge = 20
train_bench = df_bench.loc[:split_date].iloc[:-purge]
test_bench  = df_bench[df_bench.index > split_date]

X_train_bench = train_bench[["VGCI",	"VMFI",	"VFTI",	"VTRI",	"VMII"]]
y_train_bench = train_bench["systemic_stress_event"]
X_test_bench  = test_bench[["VGCI",	"VMFI",	"VFTI",	"VTRI",	"VMII"]]
y_test_bench  = test_bench["systemic_stress_event"]

In [ ]:
timesteps = 60
split_idx = pd.Timestamp("2023-03-01")
X_train_lstm_bench = X_train_bench[X_train_bench.index < split_idx].iloc[:-purge]
y_train_lstm_bench = y_train_bench[y_train_bench.index < split_idx].iloc[:-purge]
X_val_lstm_bench = X_train_bench.loc[split_idx:]
y_val_lstm_bench = y_train_bench.loc[split_idx:]

X_train_values_bench = X_train_lstm_bench.values
y_train_values_bench = y_train_lstm_bench.values
X_test_values_bench = X_test_bench.values
y_test_values_bench = y_test_bench.values
X_val_values_bench = X_val_lstm_bench.values
y_val_values_bench = y_val_lstm_bench.values
X_train_seq_bench = []
y_train_seq_bench = []
X_test_seq_bench = []
y_test_seq_bench = []
X_val_seq_bench = []
y_val_seq_bench = []
for i in range(timesteps, len(X_train_values_bench)):
  X_train_seq_bench.append(X_train_values_bench[i - timesteps:i])
  y_train_seq_bench.append(y_train_values_bench[i])
for i in range(timesteps, len(X_test_values_bench)):
  X_test_seq_bench.append(X_test_values_bench[i - timesteps:i])
  y_test_seq_bench.append(y_test_values_bench[i])
for i in range(timesteps, len(X_val_values_bench)):
  X_val_seq_bench.append(X_val_values_bench[i - timesteps:i])
  y_val_seq_bench.append(y_val_values_bench[i])

X_train_seq_bench = np.array(X_train_seq_bench)
y_train_seq_bench = np.array(y_train_seq_bench)
X_test_seq_bench = np.array(X_test_seq_bench)
y_test_seq_bench = np.array(y_test_seq_bench)
X_val_seq_bench = np.array(X_val_seq_bench)
y_val_seq_bench = np.array(y_val_seq_bench)

In [ ]:
print(X_train_seq_bench.shape,
      y_train_seq_bench.shape,
      X_val_seq_bench.shape,
      y_val_seq_bench.shape,
      X_test_seq_bench.shape,
      y_test_seq_bench.shape)

(1457, 60, 5) (1457,) (192, 60, 5) (192,) (416, 60, 5) (416,)


In [ ]:
class CombinedMetric(tf.keras.callbacks.Callback):
  def on_epoch_end(self, epoch, logs=None):
    logs = logs or {}
    recall = logs.get("val_recall", 0)
    pr_auc = logs.get("val_pr_auc", 0)
    logs["val_combined"] = 0.3 * recall + 0.7 * pr_auc

In [ ]:
classes = np.unique(y_train_seq_bench)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train_seq_bench)
class_weight = dict(zip(classes, weights))
epochs=200
batch_size=32

In [ ]:
seeds = [38, 60, 96, 146, 162]
robustness_results_bench = []
input_shape = X_train_seq_bench.shape[1:]

In [ ]:
for seed in seeds:
  os.environ["PYTHONHASHSEED"] = str(seed)
  print(f"\nRunning seed {seed}")
  tf.keras.backend.clear_session()
  random.seed(seed)
  np.random.seed(seed)
  tf.keras.utils.set_random_seed(seed)

  # LSTM MODEL
  lstm_model_bench = keras.Sequential()
  lstm_model_bench.add(layers.LSTM(128, return_sequences=True, input_shape=input_shape))
  lstm_model_bench.add(layers.Dropout(0.5))
  lstm_model_bench.add(layers.LSTM(128, return_sequences=True))
  lstm_model_bench.add(layers.LSTM(16, return_sequences=True))
  lstm_model_bench.add(layers.LSTM(8, return_sequences=True))
  lstm_model_bench.add(layers.Dropout(0.3))
  lstm_model_bench.add(layers.LSTM(8, return_sequences=False))
  lstm_model_bench.add(layers.Dropout(0.3))
  # Dense layers
  lstm_model_bench.add(layers.Dense(16, activation="relu"))
  lstm_model_bench.add(layers.Dense(1, activation="sigmoid"))
  # Compile
  lstm_model_bench.compile(optimizer=keras.optimizers.Adam(learning_rate=0.0001),
                           loss="binary_crossentropy",
                           metrics=[keras.metrics.AUC(name="pr_auc", curve="PR"),
                                    keras.metrics.AUC(name="roc_auc", curve="ROC"),
                                    tf.keras.metrics.Recall(name="recall"),
                                    tf.keras.metrics.Precision(name="precision")])

  combined_metric = CombinedMetric()
  es = tf.keras.callbacks.EarlyStopping(monitor="val_combined",
                                        mode="max",
                                        patience=50,
                                        restore_best_weights=True,
                                        verbose=1)

  # TRAIN
  lstm_results_bench = train_classifier(lstm_model_bench, X_train_seq_bench, y_train_seq_bench, X_test_seq_bench, y_test_seq_bench, X_val=X_val_seq_bench, y_val=y_val_seq_bench, early_stopping=True)
  lstm_recall_bench, lstm_f1_bench, lstm_roc_auc_bench, lstm_cm_bench, lstm_probs_bench, lstm_preds_bench = lstm_results_bench
  # Store results
  robustness_results_bench.append({"seed": seed,
                                   "recall": lstm_recall_bench,
                                   "f1": lstm_f1_bench,
                                   "roc_auc": lstm_roc_auc_bench,
                                   "tn": lstm_cm_bench[0,0],
                                   "fp": lstm_cm_bench[0,1],
                                   "fn": lstm_cm_bench[1,0],
                                   "tp": lstm_cm_bench[1,1]})


Running seed 38


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 26s 230ms/step - loss: 0.6944 - pr_auc: 0.0657 - precision: 0.0781 - recall: 0.9741 - roc_auc: 0.4400 - val_loss: 0.7017 - val_pr_auc: 0.0521 - val_precision: 0.0521 - val_recall: 1.0000 - val_roc_auc: 0.5000 - val_combined: 0.3365
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 197ms/step - loss: 0.6919 - pr_auc: 0.1137 - precision: 0.0806 - recall: 1.0000 - roc_auc: 0.5816 - val_loss: 0.7037 - val_pr_auc: 0.0588 - val_precision: 0.0521 - val_recall: 1.0000 - val_roc_auc: 0.5885 - val_combined: 0.3412
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 193ms/step - loss: 0.6903 - pr_auc: 0.1571 - precision: 0.0801 - recall: 0.9914 - roc_auc: 0.6002 - val_loss: 0.7061 - val_pr_auc: 0.0799 - val_precision: 0.0529 - val_recall: 1.0000 - val_roc_auc: 0.7033 - val_combined: 0.3560
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 207ms/step - loss: 0.6887 - pr_auc: 0.1350 - precision: 0.0811 - recall: 1.0000 - roc_auc: 0.6450 - val_loss: 0.7170 - val_pr_auc: 0.0701 - val_pr

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 209ms/step - loss: 0.6943 - pr_auc: 0.0720 - precision: 0.0603 - recall: 0.1638 - roc_auc: 0.4686 - val_loss: 0.6861 - val_pr_auc: 0.0712 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6049 - val_combined: 0.0499
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 186ms/step - loss: 0.6901 - pr_auc: 0.1199 - precision: 0.1325 - recall: 0.3448 - roc_auc: 0.6271 - val_loss: 0.6842 - val_pr_auc: 0.0837 - val_precision: 0.0769 - val_recall: 0.4000 - val_roc_auc: 0.6973 - val_combined: 0.1786
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 198ms/step - loss: 0.6903 - pr_auc: 0.1130 - precision: 0.1367 - recall: 0.3534 - roc_auc: 0.5960 - val_loss: 0.6749 - val_pr_auc: 0.1074 - val_precision: 0.0385 - val_recall: 0.1000 - val_roc_auc: 0.7885 - val_combined: 0.1052
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 189ms/step - loss: 0.6868 - pr_auc: 0.1841 - precision: 0.1957 - recall: 0.3879 - roc_auc: 0.6534 - val_loss: 0.6586 - val_pr_auc: 0.1139 -

...

Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 18s 222ms/step - loss: 0.6983 - pr_auc: 0.0521 - precision: 0.0375 - recall: 0.0259 - roc_auc: 0.3001 - val_loss: 0.6737 - val_pr_auc: 0.0559 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.5824 - val_combined: 0.0391
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 196ms/step - loss: 0.6941 - pr_auc: 0.0752 - precision: 0.0000e+00 - recall: 0.0000e+00 - roc_auc: 0.4869 - val_loss: 0.6717 - val_pr_auc: 0.0613 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6236 - val_combined: 0.0429
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 179ms/step - loss: 0.6920 - pr_auc: 0.1170 - precision: 0.0769 - recall: 0.0172 - roc_auc: 0.5568 - val_loss: 0.6700 - val_pr_auc: 0.0648 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.6374 - val_combined: 0.0453
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 188ms/step - loss: 0.6897 - pr_auc: 0.1640 - precision: 0.3200 - recall: 0.0690 - roc_auc: 0.6300 - val_loss: 0.665

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 19s 223ms/step - loss: 0.6942 - pr_auc: 0.0668 - precision: 0.0609 - recall: 0.2241 - roc_auc: 0.4332 - val_loss: 0.6933 - val_pr_auc: 0.0399 - val_precision: 0.0000e+00 - val_recall: 0.0000e+00 - val_roc_auc: 0.4093 - val_combined: 0.0279
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 197ms/step - loss: 0.6916 - pr_auc: 0.1070 - precision: 0.1230 - recall: 0.3362 - roc_auc: 0.5744 - val_loss: 0.6985 - val_pr_auc: 0.0603 - val_precision: 0.1111 - val_recall: 1.0000 - val_roc_auc: 0.6154 - val_combined: 0.3422
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 175ms/step - loss: 0.6886 - pr_auc: 0.1204 - precision: 0.0979 - recall: 0.5345 - roc_auc: 0.5929 - val_loss: 0.7068 - val_pr_auc: 0.0815 - val_precision: 0.1000 - val_recall: 1.0000 - val_roc_auc: 0.7198 - val_combined: 0.3570
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 9s 197ms/step - loss: 0.6733 - pr_auc: 0.2388 - precision: 0.1354 - recall: 0.6293 - roc_auc: 0.6979 - val_loss: 0.7519 - val_pr_auc: 0.0832 -

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 18s 197ms/step - loss: 0.6949 - pr_auc: 0.0633 - precision: 0.0781 - recall: 0.9397 - roc_auc: 0.4089 - val_loss: 0.7026 - val_pr_auc: 0.0308 - val_precision: 0.0521 - val_recall: 1.0000 - val_roc_auc: 0.2121 - val_combined: 0.3216
Epoch 2/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 10s 210ms/step - loss: 0.6934 - pr_auc: 0.0753 - precision: 0.0807 - recall: 0.8707 - roc_auc: 0.5098 - val_loss: 0.6974 - val_pr_auc: 0.0909 - val_precision: 0.0538 - val_recall: 1.0000 - val_roc_auc: 0.7253 - val_combined: 0.3636
Epoch 3/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 21s 465ms/step - loss: 0.6921 - pr_auc: 0.0897 - precision: 0.0903 - recall: 0.8621 - roc_auc: 0.5475 - val_loss: 0.6949 - val_pr_auc: 0.0395 - val_precision: 0.0761 - val_recall: 0.7000 - val_roc_auc: 0.3956 - val_combined: 0.2376
Epoch 4/200
46/46 ━━━━━━━━━━━━━━━━━━━━ 8s 175ms/step - loss: 0.6917 - pr_auc: 0.1054 - precision: 0.0856 - recall: 0.7241 - roc_auc: 0.5895 - val_loss: 0.6951 - val_pr_auc: 0.0550 - val_p

In [ ]:
df_robustness_bench = pd.DataFrame(robustness_results_bench)
df_robustness_bench

,seed,recall,f1,roc_auc,tn,fp,fn,tp
0,38,0.0,0.0,0.640838,339,53,24,0
1,60,0.0,0.0,0.358844,363,29,24,0
2,96,0.0,0.0,0.293155,305,87,24,0
3,146,0.0,0.0,0.402742,381,11,24,0
4,162,0.0,0.0,0.272853,369,23,24,0


```python
df_robustness_bench.to_csv("df_robustness_bench.csv", index=True)
files.download("df_robustness_bench.csv")
df_robustness_bench.to_parquet("df_robustness_bench.parquet", index=True)
files.download("df_robustness_bench.parquet")
```

In [ ]:
#df_robustness_bench = import_data(file_path)

In [ ]:
df_robustness_bench = pd.concat([df_robustness_bench, pd.DataFrame({'seed': 23,
                                                                    'recall': 0,
                                                                    'f1': 0,
                                                                    'roc_auc': 0.48,
                                                                    'tn': 369,
                                                                    'fp': 23,
                                                                    'fn': 24,
                                                                    'tp': 0},
                                                                    index=[0])], ignore_index=True) #add results from baseline model
df_robustness_bench

,seed,recall,f1,roc_auc,tn,fp,fn,tp
0,38,0.0,0.0,0.640838,339,53,24,0
1,60,0.0,0.0,0.358844,363,29,24,0
2,96,0.0,0.0,0.293155,305,87,24,0
3,146,0.0,0.0,0.402742,381,11,24,0
4,162,0.0,0.0,0.272853,369,23,24,0
5,23,0.0,0.0,0.480000,369,23,24,0


In [ ]:
df_robustness_bench = df_robustness_bench.set_index('seed')[['recall',	'f1',	'roc_auc']].sort_index()
df_robustness_bench = df_robustness_bench.rename(columns={'recall':'Recall',	'f1':'F1-Score',	'roc_auc':'ROC-AUC'})

In [ ]:
df_robustness_bench.round(2)

,Recall,F1-Score,ROC-AUC
seed,,,
23,0.0,0.0,0.48
38,0.0,0.0,0.64
60,0.0,0.0,0.36
96,0.0,0.0,0.29
146,0.0,0.0,0.40
162,0.0,0.0,0.27


In [ ]:
df_robustness_bench.mean().round(2).rename('mean')

,mean
Recall,0.00
F1-Score,0.00
ROC-AUC,0.41


In [ ]:
df_robustness_bench.std().round(2).rename('std')

,std
Recall,0.00
F1-Score,0.00
ROC-AUC,0.14


We refer to the project report for a discussion on the results of the LSTM sensitivity analysis to random initialization.